In [ ]:
import os

if not os.path.exists("static"):
    os.makedirs("static")

print("✅ 'static' folder created!")


In [ ]:
from flask import Flask, render_template_string, request, redirect, send_file
import pandas as pd
import matplotlib.pyplot as plt

app = Flask(__name__)
portfolio = []

# Simple inline HTML (no templates needed)
html_template = '''
<!DOCTYPE html>
<html>
<head><title>Stock App</title></head>
<body style="font-family:sans-serif; padding:30px;">
    <h2>📈 Add Stock</h2>
    <form method="post" action="/add">
        <input name="ticker" placeholder="Ticker" required><br>
        <input name="buy_price" placeholder="Buy Price" type="number" step="0.01" required><br>
        <input name="quantity" placeholder="Quantity" type="number" required><br>
        <input name="current_price" placeholder="Current Price" type="number" step="0.01" required><br>
        <button type="submit">Add</button>
    </form><br>

    <form method="post" action="/remove">
        <input name="ticker" placeholder="Ticker to Remove"><br>
        <button type="submit">Remove</button>
    </form><br>

    <form method="post" action="/update">
        <input name="ticker" placeholder="Ticker to Update"><br>
        <input name="quantity" placeholder="New Quantity" type="number"><br>
        <input name="current_price" placeholder="New Price" type="number"><br>
        <button type="submit">Update</button>
    </form><br>

    <form method="post" action="/search">
        <input name="ticker" placeholder="Search Ticker"><br>
        <button type="submit">Search</button>
    </form><br>

    <a href="/summary">Summary</a> |
    <a href="/sort">Sort</a> |
    <a href="/chart">Chart</a> |
    <a href="/export_csv">CSV</a> |
    <a href="/export_pdf">PDF</a>

    <h3>Portfolio</h3>
    <ul>
        {% for stock in portfolio %}
        <li>{{ stock['ticker'] }} — ₹{{ stock['buy_price'] }} → ₹{{ stock['current_price'] }} (Qty: {{ stock['quantity'] }})</li>
        {% endfor %}
    </ul>
</body>
</html>
'''

@app.route('/')
def index():
    return render_template_string(html_template, portfolio=portfolio)

@app.route('/add', methods=['POST'])
def add_stock():
    portfolio.append({
        'ticker': request.form['ticker'].upper(),
        'buy_price': float(request.form['buy_price']),
        'quantity': int(request.form['quantity']),
        'current_price': float(request.form['current_price'])
    })
    return redirect('/')

@app.route('/remove', methods=['POST'])
def remove_stock():
    ticker = request.form['ticker'].upper()
    global portfolio
    portfolio = [stock for stock in portfolio if stock['ticker'] != ticker]
    return redirect('/')

@app.route('/update', methods=['POST'])
def update_stock():
    ticker = request.form['ticker'].upper()
    for stock in portfolio:
        if stock['ticker'] == ticker:
            stock['quantity'] = int(request.form['quantity'])
            stock['current_price'] = float(request.form['current_price'])
    return redirect('/')

@app.route('/search', methods=['POST'])
def search_stock():
    ticker = request.form['ticker'].upper()
    for stock in portfolio:
        if stock['ticker'] == ticker:
            return f"<h2>{ticker}</h2><p>Qty: {stock['quantity']} | Current Price: ₹{stock['current_price']}</p><a href='/'>Back</a>"
    return "<p>Stock not found.</p><a href='/'>Back</a>"

@app.route('/summary')
def summary():
    invested = sum(s['buy_price'] * s['quantity'] for s in portfolio)
    current = sum(s['current_price'] * s['quantity'] for s in portfolio)
    profit = current - invested
    return f"<h2>Summary</h2><p>Total Invested: ₹{invested}<br>Current Value: ₹{current}<br>Profit/Loss: ₹{profit}</p><a href='/'>Back</a>"

@app.route('/sort')
def sort_portfolio():
    sorted_port = sorted(portfolio, key=lambda s: ((s['current_price'] - s['buy_price']) / s['buy_price']), reverse=True)
    html = "<h2>Sorted by Profit %</h2><ul>"
    for s in sorted_port:
        html += f"<li>{s['ticker']}: Profit ₹{(s['current_price'] - s['buy_price']) * s['quantity']}</li>"
    html += "</ul><a href='/'>Back</a>"
    return html

@app.route('/export_csv')
def export_csv():
    df = pd.DataFrame(portfolio)
    path = 'portfolio.csv'
    df.to_csv(path, index=False)
    return send_file(path, as_attachment=True)

@app.route('/export_pdf')
def export_pdf():
    df = pd.DataFrame(portfolio)
    fig, ax = plt.subplots()
    ax.axis('off')
    ax.table(cellText=df.values, colLabels=df.columns, loc='center')
    path = 'portfolio.pdf'
    fig.savefig(path)
    return send_file(path, as_attachment=True)

@app.route('/chart')
def chart():
    if not portfolio:
        return "<p>No data</p><a href='/'>Back</a>"
    labels = [s['ticker'] for s in portfolio]
    sizes = [s['quantity'] * s['current_price'] for s in portfolio]
    fig, ax = plt.subplots()
    ax.pie(sizes, labels=labels, autopct='%1.1f%%')
    path = 'static/chart.png'
    fig.savefig(path)
    return f"<h2>Chart</h2><img src='/{path}'><br><a href='/'>Back</a>"

# Start Flask app
app.run()


In [ ]:
import IPython.display as display

# Start Flask app
display.display(display.Markdown("[Click here to open the app](http://127.0.0.1:5000)"))

app.run()
